In [ ]:
!pip install arch
!pip install pytorch-lightning

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import yfinance as yf

from sklearn.preprocessing import StandardScaler
import pytorch_lightning as pl
from pytorch_lightning import Trainer, seed_everything
from pytorch_lightning.loggers import CSVLogger
from torch.utils.data import Dataset, DataLoader
from scipy.stats import norm
from arch.univariate import SkewStudent

# Check if CUDA is available
print(torch.cuda.is_available())  # Should return True

# Add these imports at the top of the file
from datetime import datetime, timedelta

# Rest of the imports remain the same
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import yfinance as yf
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from torch.utils.data import Dataset, DataLoader
from scipy.stats import norm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 985.1/985.1 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.2/815.2 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 926.4/926.4 kB 54.5 MB/s eta 0:00:00
True


# Steps for Simulation

## 1. Retrieve Estimated Parameters
Extract the predicted parameters from GARCHNet:
- Volatility ($\sigma_t^2$)
- Skewness ($\lambda_t$)
- Degrees of freedom ($\nu_t$)

## 2. Simulate Residuals
Generate standardized residuals ($z_t$) using Hansen's Skewed T-distribution:
$$z_t \sim \text{Skewed-T}(\lambda_t, \nu_t)$$

## 3. Compute Returns
Scale the residuals by the predicted volatility:
$$r_t = \sqrt{\sigma_t^2} \cdot z_t$$

## 4. Calculate Price Path
Starting from initial price $P_0$, compute the price path:
$$P_t = P_{t-1} \cdot e^{r_t}$$

This simulation process allows us to generate multiple price paths that incorporate both the predicted volatility and the skewed, heavy-tailed nature of financial returns through Hansen's Skewed T-distribution.

In [ ]:
import torch
import torch.nn as nn
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from pytorch_lightning import Trainer, seed_everything
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from arch.univariate import SkewStudent

class ParallelTimeseriesDataset(Dataset):
    def __init__(self, X: torch.Tensor, y: torch.Tensor, seq_len: int = 1, num_parallel: int = 10):
        self.X = X.cpu()
        self.y = y.cpu()
        self.seq_len = seq_len
        self.num_parallel = num_parallel

    def __len__(self):
        return (len(self.X) - self.seq_len - self.num_parallel) // self.num_parallel

    def __getitem__(self, index):
        start_idx = index * self.num_parallel
        sequences, targets = [], []
        for i in range(self.num_parallel):
            seq_start = start_idx + i
            seq_end = seq_start + self.seq_len
            sequences.append(self.X[seq_start:seq_end])
            targets.append(self.y[seq_end - 1])
        return torch.stack(sequences), torch.stack(targets)

class ParallelDataModule(pl.LightningDataModule):
    def __init__(self, df, training_length=1000, seq_len=20, num_parallel=10, batch_size=32):
        super().__init__()
        self.df = df
        self.training_length = training_length
        self.seq_len = seq_len
        self.num_parallel = num_parallel
        self.batch_size = batch_size
        self.preprocessing = StandardScaler()
        self.test_case = 0

    def setup(self, stage=None):
        data = self.df.iloc[self.test_case:self.training_length + self.test_case]
        X = data['log_returns'].values[:self.training_length]
        y = data['log_returns'].shift(-1).values[:self.training_length]

        X_scaled = self.preprocessing.fit_transform(X.reshape(-1, 1)).flatten()
        y_scaled = self.preprocessing.transform(y.reshape(-1, 1)).flatten()

        X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
        y_tensor = torch.tensor(y_scaled, dtype=torch.float32)

        self.train_dataset = ParallelTimeseriesDataset(X_tensor, y_tensor, seq_len=self.seq_len, num_parallel=self.num_parallel)

        self.X_test = torch.tensor(
            self.preprocessing.transform(
                data['log_returns'].values[-self.seq_len - 1:-1].reshape(-1, 1)
            ).flatten(),
            dtype=torch.float32
        ).unsqueeze(0)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=False, num_workers=4, pin_memory=True)

    def move_timestep(self):
        self.test_case += 1

    def gather_prediction(self, prediction):
        current_position = self.test_case + self.training_length
        self.df.iloc[current_position, self.df.columns.get_loc('VaR')] = prediction

        current_date = self.df.index[current_position]
        current_return = self.df.iloc[current_position]['log_returns']

        print(f"Prediction {self.test_case + 1}")
        print(f"Date: {current_date}")
        print(f"Log Return: {current_return:.6f}")
        print(f"VaR: {float(prediction):.6f}")
        print("-" * 50)

class ParallelGARCHNet(pl.LightningModule):
    def __init__(self, n_features, hidden_size, seq_len, num_parallel, num_layers=1, dropout=0.0, learning_rate=3e-4):
        super().__init__()
        self.save_hyperparameters()

        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden_size, num_layers=num_layers, dropout=dropout, batch_first=True)
        self.linear = nn.Linear(hidden_size, 64)
        self.linear2 = nn.Linear(64, 32)
        self.linear3 = nn.Linear(32, 1)
        self.softplus = nn.Softplus()

        self.linear3_1 = nn.Linear(32, 1)  # Volatility
        self.linear3_2 = nn.Linear(32, 1)  # Skewness
        self.linear3_3 = nn.Linear(32, 1)  # Degrees of freedom

        self.tanh = nn.Tanh()
        self.relu = nn.ReLU()

    def forward(self, x):
        batch_size, num_parallel, seq_len, n_features = x.shape
        x = x.view(batch_size * num_parallel, seq_len, n_features)

        lstm_out, _ = self.lstm(x)
        last_hidden = lstm_out[:, -1]

        y_pred = self.linear(last_hidden)
        y_pred = self.linear2(y_pred)

        vol = self.softplus(self.linear3_1(y_pred))
        skew = self.tanh(self.linear3_2(y_pred))
        df = self.relu(self.linear3_3(y_pred)) + 2.05

        return torch.cat([vol, skew, df], dim=1)

    def skewed_t_loss(self, y, pred):
        vol, skew, df = pred[:, 0], pred[:, 1], pred[:, 2]
        c = torch.lgamma((df + 1)/2) - torch.lgamma(df/2) - 0.5 * torch.log(np.pi * (df - 2))
        a = 4 * skew * torch.exp(c) * (df - 2) / (df - 1)
        b = torch.sqrt(1 + 3 * skew**2 - a**2)
        z = y / torch.sqrt(vol)
        ll = torch.log(b) + c - 0.5 * torch.log(vol) - ((df + 1)/2) * torch.log(1 + ((b * z + a) / (1 + skew.sign() * skew))**2 / (df - 2))
        return -torch.mean(ll)

    def training_step(self, batch, batch_idx):
        x, y = batch
        pred = self(x)
        loss = self.skewed_t_loss(y, pred)
        self.log('train_loss', loss)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)
        return {'optimizer': optimizer, 'lr_scheduler': scheduler, 'monitor': 'train_loss'}

# Prepare data and train
if __name__ == "__main__":
    seed_everything(42)

    # Example DataFrame setup
    nikkei = yf.download('^N225', start='2000-01-01')
    data = nikkei[['Close']].copy()
    data['log_returns'] = np.log(data['Close'] / data['Close'].shift(1))
    data = data.dropna()

    data_module = ParallelDataModule(data, training_length=1000, seq_len=20, num_parallel=10, batch_size=32)

    model = ParallelGARCHNet(
        n_features=1,
        hidden_size=100,
        seq_len=20,
        num_parallel=10,
        num_layers=1,
        dropout=0.0,
        learning_rate=3e-4
    )

    trainer = Trainer(max_epochs=100, accelerator='gpu', devices=1, callbacks=[EarlyStopping(monitor='train_loss', patience=10), LearningRateMonitor(logging_interval='epoch')])
    trainer.fit(model, data_module)


INFO:lightning_fabric.utilities.seed:Seed set to 42
[*********************100%***********************]  1 of 1 completed
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name      | Type     | Params | Mode 
-----------------------------------------------
0 | lstm      | LSTM     | 41.2 K | train
1 | linear    | Linear   | 6.5 K  | train
2 | linear2   | Linear   | 2.1 K  | train
3 | linear3   | Linear   | 33     | train
4 | softplus  | Softplus | 0      | train
5 | linear3_1 | Linear   | 33     | train
6 | linear3_2 | Linear   | 33     | train
7 | linear3_3 | Linear   | 33     | train
8 | tanh      | Tanh     | 0      | train
9 | relu      | ReLU     | 0      |

Training: |          | 0/? [00:00<?, ?it/s]

ValueError: not enough values to unpack (expected 4, got 3)

In [ ]:
# Initialize model and data module
model = ParallelGARCHNet(
    n_features=1,
    hidden_size=100,
    seq_len=20,
    num_parallel=10  # Process 10 days at once
)

data_module = ParallelDataModule(
    df=your_data,
    training_length=1000,
    seq_len=20,
    num_parallel=10,
    batch_size=32
)

# Train model
trainer = pl.Trainer(
    max_epochs=100,
    accelerator='gpu',
    devices=1
)

trainer.fit(model, data_module)

# Make predictions
predictions = model.predict_var(test_data)

NameError: name 'your_data' is not defined

In [ ]:
import torch
import torch.nn as nn
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from pytorch_lightning import Trainer, seed_everything
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from torch.optim.lr_scheduler import ReduceLROnPlateau
from arch.univariate import SkewStudent

class ParallelTimeseriesDataset(Dataset):
    def __init__(self, X: torch.Tensor, y: torch.Tensor, seq_len: int = 1, num_parallel: int = 10):
        self.X = X.cpu()
        self.y = y.cpu()
        self.seq_len = seq_len
        self.num_parallel = num_parallel

    def __len__(self):
        return (len(self.X) - self.seq_len - self.num_parallel) // self.num_parallel

    def __getitem__(self, index):
        start_idx = index * self.num_parallel
        sequences, targets = [], []
        for i in range(self.num_parallel):
            seq_start = start_idx + i
            seq_end = seq_start + self.seq_len
            sequences.append(self.X[seq_start:seq_end])
            targets.append(self.y[seq_end - 1])
        return torch.stack(sequences), torch.stack(targets)

class ParallelDataModule(pl.LightningDataModule):
    def __init__(self, df, training_length=1000, seq_len=20, num_parallel=10, batch_size=32):
        super().__init__()
        self.df = df
        self.training_length = training_length
        self.seq_len = seq_len
        self.num_parallel = num_parallel
        self.batch_size = batch_size
        self.preprocessing = StandardScaler()
        self.test_case = 0

    def setup(self, stage=None):
        data = self.df.iloc[self.test_case:self.training_length + self.test_case]
        X = data['log_returns'].values[:self.training_length]
        y = data['log_returns'].shift(-1).values[:self.training_length]

        X_scaled = self.preprocessing.fit_transform(X.reshape(-1, 1)).flatten()
        y_scaled = self.preprocessing.transform(y.reshape(-1, 1)).flatten()

        X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
        y_tensor = torch.tensor(y_scaled, dtype=torch.float32)

        self.train_dataset = ParallelTimeseriesDataset(X_tensor, y_tensor, seq_len=self.seq_len, num_parallel=self.num_parallel)

        self.X_test = torch.tensor(
            self.preprocessing.transform(
                data['log_returns'].values[-self.seq_len - 1:-1].reshape(-1, 1)
            ).flatten(),
            dtype=torch.float32
        ).unsqueeze(0)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=False, num_workers=4, pin_memory=True)

    def move_timestep(self):
        self.test_case += 1

    def gather_prediction(self, prediction):
        current_position = self.test_case + self.training_length
        self.df.iloc[current_position, self.df.columns.get_loc('VaR')] = prediction

        current_date = self.df.index[current_position]
        current_return = self.df.iloc[current_position]['log_returns']

        print(f"Prediction {self.test_case + 1}")
        print(f"Date: {current_date}")
        print(f"Log Return: {current_return:.6f}")
        print(f"VaR: {float(prediction):.6f}")
        print("-" * 50)

class ParallelGARCHNet(pl.LightningModule):
    def __init__(self, n_features, hidden_size, seq_len, num_parallel, num_layers=1, dropout=0.0, learning_rate=3e-4):
        super().__init__()
        self.save_hyperparameters()

        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden_size, num_layers=num_layers, dropout=dropout, batch_first=True)
        self.linear = nn.Linear(hidden_size, 64)
        self.linear2 = nn.Linear(64, 32)
        self.linear3 = nn.Linear(32, 1)
        self.softplus = nn.Softplus()

        self.linear3_1 = nn.Linear(32, 1)  # Volatility
        self.linear3_2 = nn.Linear(32, 1)  # Skewness
        self.linear3_3 = nn.Linear(32, 1)  # Degrees of freedom

        self.tanh = nn.Tanh()
        self.relu = nn.ReLU()

    def forward(self, x):
        batch_size, num_parallel, seq_len, n_features = x.shape
        x = x.view(batch_size * num_parallel, seq_len, n_features)

        lstm_out, _ = self.lstm(x)
        last_hidden = lstm_out[:, -1]

        y_pred = self.linear(last_hidden)
        y_pred = self.linear2(y_pred)

        vol = self.softplus(self.linear3_1(y_pred))
        skew = self.tanh(self.linear3_2(y_pred))
        df = self.relu(self.linear3_3(y_pred)) + 2.05

        return torch.cat([vol, skew, df], dim=1)

    def skewed_t_loss(self, y, pred):
        vol, skew, df = pred[:, 0], pred[:, 1], pred[:, 2]
        c = torch.lgamma((df + 1)/2) - torch.lgamma(df/2) - 0.5 * torch.log(np.pi * (df - 2))
        a = 4 * skew * torch.exp(c) * (df - 2) / (df - 1)
        b = torch.sqrt(1 + 3 * skew**2 - a**2)
        z = y / torch.sqrt(vol)
        ll = torch.log(b) + c - 0.5 * torch.log(vol) - ((df + 1)/2) * torch.log(1 + ((b * z + a) / (1 + skew.sign() * skew))**2 / (df - 2))
        return -torch.mean(ll)

    def training_step(self, batch, batch_idx):
        x, y = batch
        pred = self(x)
        loss = self.skewed_t_loss(y, pred)
        self.log('train_loss', loss)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
        scheduler = {
            'scheduler': ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5, verbose=True),
            'monitor': 'train_loss'
        }
        return [optimizer], [scheduler]

# Experiment function for training and evaluation
def experiment_nikkei(model_name='garch_skew'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    nikkei = yf.download('^N225', start='2000-01-01')
    data = nikkei[['Close']].copy()
    data['log_returns'] = np.log(data['Close'] / data['Close'].shift(1))
    data = data.dropna()
    data['VaR'] = np.nan

    memory_sizes = [20]
    sample_dates = [data.index[int(len(data) * 0.7)].strftime('%Y-%m-%d')]

    for mem_size in memory_sizes:
        p = dict(
            training_length=1024,
            seq_len=mem_size,
            batch_size=32,
            max_epochs=100,
            n_features=1,
            hidden_size=100,
            num_layers=1,
            dropout=0,
            learning_rate=3e-4,
            num_train=150
        )

        for sample_start in sample_dates:
            current_data = data.loc[(data.index > sample_start)].copy()

            data_module = ParallelDataModule(
                df=current_data[['log_returns', 'VaR']],
                training_length=p['training_length'],
                seq_len=p['seq_len'],
                batch_size=p['batch_size'],
                num_parallel=10
            )

            model = ParallelGARCHNet(
                n_features=p['n_features'],
                hidden_size=p['hidden_size'],
                seq_len=p['seq_len'],
                num_parallel=10,
                num_layers=p['num_layers'],
                dropout=p['dropout'],
                learning_rate=p['learning_rate']
            ).to(device)

            trainer = Trainer(
                max_epochs=p['max_epochs'],
                accelerator='gpu' if torch.cuda.is_available() else 'cpu',
                devices=1,
                callbacks=[EarlyStopping(monitor='train_loss', patience=10), LearningRateMonitor(logging_interval='epoch')]
            )

            trainer.fit(model, data_module)

if __name__ == "__main__":
    seed_everything(42)
    experiment_nikkei('garch_skew')


INFO:lightning_fabric.utilities.seed:Seed set to 42


Using device: cuda


[*********************100%***********************]  1 of 1 completed
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name      | Type     | Params | Mode 
-----------------------------------------------
0 | lstm      | LSTM     | 41.2 K | train
1 | linear    | Linear   | 6.5 K  | train
2 | linear2   | Linear   | 2.1 K  | train
3 | linear3   | Linear   | 33     | train
4 | softplus  | Softplus | 0      | train
5 | linear3_1 | Linear   | 33     | train
6 | linear3_2 | Linear   | 33     | train
7 | linear3_3 | Linear   | 33     | train
8 | tanh      | Tanh     | 0      | train
9 | relu      | ReLU     | 0      | train
---------------------------------------------

Training: |          | 0/? [00:00<?, ?it/s]

ValueError: not enough values to unpack (expected 4, got 3)

In [ ]:
class ParallelDataModule(pl.LightningDataModule):
    def __init__(self, df, training_length=1000, seq_len=20, num_parallel=10, batch_size=32, device=None):
        super().__init__()
        self.df = df
        self.test_case = 0
        self.training_length = training_length
        self.seq_len = seq_len
        self.num_parallel = num_parallel
        self.batch_size = batch_size
        self.preprocessing = StandardScaler()
        self.device = device if device is not None else torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def setup_train(self):
        # Get training data slice
        data = self.df.iloc[self.test_case:self.training_length + self.test_case]

        # Prepare features and targets
        X = data[['log_returns']].values
        y = data[['log_returns']].shift(-1).values

        # Scale the data
        X = self.preprocessing.fit_transform(X)
        y = self.preprocessing.transform(y)

        # Create dataset
        self.train_dataset = ParallelTimeseriesDataset(
            X[:-1], y[:-1],
            seq_len=self.seq_len,
            num_parallel=self.num_parallel
        )

        # Prepare test data
        self.X_test = torch.tensor(
            self.preprocessing.transform(
                data[['log_returns']].values[-self.seq_len-self.num_parallel:-1]
            ),
            dtype=torch.float32,
            device=self.device
        ).unsqueeze(0)

In [ ]:
# Initialize data module
dm = ParallelDataModule(
    df=current_data[['log_returns', 'VaR']],
    training_length=p['training_length'],
    seq_len=p['seq_len'],
    num_parallel=num_parallel,
    batch_size=p['batch_size'],
    device=device  # Pass the device
)

NameError: name 'current_data' is not defined

In [ ]:
# def experiment_nikkei(model_name='garch_skew'):
#     # Set up device and print info
#     device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#     print(f"Using device: {device}")

#     # Model setup with proper configuration
#     if model_name == 'garch_skew':
#         model_class = SkewedGARCHVaRNet
#         dist = SkewStudent
#         loss = hansen_garch_skewed_student_loss
#     elif model_name == 'garch_norm':
#         model_class = GARCHVaRNet
#         dist = norm
#         loss = garch_normal_loss
#     elif model_name == 'caviar':
#         model_class = VaRNet
#         dist = None
#         loss = caviar_loss
#     elif model_name == 'caviar_huber':
#         model_class = VaRNet
#         dist = None
#         loss = huber_loss
#     else:
#         raise ValueError(f"Unknown model name: {model_name}")

#     # Download and prepare Nikkei data
#     nikkei = yf.download('^N225', start='2000-01-01')
#     data = nikkei[['Close']].copy()
#     data['log_returns'] = np.log(data['Close'] / data['Close'].shift(1))
#     data = data.dropna()
#     data['VaR'] = np.nan

#     # Training parameters setup
#     memory_sizes = [50]
#     total_samples = len(data)
#     sample_points = [int(total_samples * 0.2)]
#     sample_dates = [data.index[point].strftime('%Y-%m-%d') for point in sample_points]

#     # Device-aware data module with proper initialization
#     class DeviceAwareDataModule(ValueAtRiskDataModule):
#         def __init__(self, *args, device=None, **kwargs):
#             super().__init__(*args, **kwargs)
#             self.device = device if device is not None else torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#         def setup_train(self):
#             super().setup_train()
#             if self.X_train is not None:
#                 self.X_train = self.X_train.to(self.device)
#             if self.y_train is not None:
#                 self.y_train = self.y_train.to(self.device)
#             if self.X_test is not None:
#                 self.X_test = self.X_test.to(self.device)

#     # Training loop
#     for mem_size in memory_sizes:
#         # Training parameters
#         p = dict(
#             training_length=1024,
#             seq_len=mem_size,
#             batch_size=1024,
#             criterion=loss,
#             max_epochs=100,
#             n_features=1,
#             hidden_size=100,
#             num_layers=1,
#             dropout=0,
#             learning_rate=3e-4,
#             num_train=50
#         )

#         for sample_start in sample_dates:
#             # Prepare data for current sample
#             current_data = data.loc[(data.index > sample_start)].copy()

#             # Initialize data module
#             dm = DeviceAwareDataModule(
#                 df=current_data[['log_returns', 'VaR']],
#                 training_length=p['training_length'],
#                 seq_len=p['seq_len'],
#                 batch_size=p['batch_size'],
#                 num_workers=8,
#                 device=device
#             )

#             for test_case in range(p['num_train']):
#                 print(f"Processing test case {test_case + 1}/{p['num_train']}")

#                 # Initialize and configure model
#                 model = model_class(
#                     n_features=p['n_features'],
#                     hidden_size=p['hidden_size'],
#                     seq_len=p['seq_len'],
#                     batch_size=p['batch_size'],
#                     criterion=p['criterion'],
#                     num_layers=p['num_layers'],
#                     dropout=p['dropout'],
#                     learning_rate=p['learning_rate'],
#                     dist=dist
#                 ).to(device)

#                 # Configure trainer
#                 trainer = Trainer(
#                     max_epochs=p['max_epochs'],
#                     accelerator='gpu' if torch.cuda.is_available() else 'cpu',
#                     devices=1,
#                     enable_progress_bar=True,
#                     log_every_n_steps=1,
#                     enable_model_summary=True,
#                     enable_checkpointing=False
#                 )

#                 # Setup and train
#                 dm.setup_train()
#                 trainer.fit(model, dm)

#                 # Make predictions
#                 try:
#                     with torch.no_grad():
#                         prediction = model.predict_var(dm.X_test)

#                         # Ensure prediction is properly formatted
#                         if isinstance(prediction, torch.Tensor):
#                             prediction = prediction.cpu().numpy()
#                         prediction = np.array(prediction).reshape(-1, 1)

#                         # Transform prediction back to original scale
#                         transformed_prediction = dm.preprocessing.inverse_transform(prediction)[0][0]

#                         # Store prediction
#                         dm.gather_prediction(transformed_prediction)

#                 except Exception as e:
#                     print(f"Error during prediction: {str(e)}")
#                     continue

#                 # Move to next timestep
#                 dm.move_timestep()

#             # Save results for current sample
#             output_filename = f'nikkei_{model_name}_{sample_start}_{mem_size}.csv'
#             dm.df[['log_returns', 'VaR']].to_csv(output_filename)
#             print(f"Results saved to {output_filename}")

# if __name__ == "__main__":
#     seed_everything(42)  # Set seed for reproducibility
#     experiment_nikkei('garch_skew')

In [ ]:
class SkewedGARCHVaRNet(GARCHVaRNet):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # Additional layers for skewed distribution
        self.linear3_1 = nn.Linear(32, 1)  # Volatility
        self.linear3_2 = nn.Linear(32, 1)  # Skewness
        self.linear3_3 = nn.Linear(32, 1)  # Degrees of freedom

        self.tanh = nn.Tanh()
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.to(self.device)
        lstm_out, _ = self.lstm(x)
        y_pred = self.linear(lstm_out[:, -1])
        y_pred = self.linear2(y_pred)

        vol = self.softplus(self.linear3_1(y_pred))
        skew = self.tanh(self.linear3_2(y_pred))
        df = self.relu(self.linear3_3(y_pred)) + 2.05

        return torch.cat([vol, skew, df], dim=1)

    def predict_var(self, x):
        """
        Predict VaR and return all parameters
        Returns:
            dict: Dictionary containing VaR and all parameters
        """
        self.eval()
        with torch.no_grad():
            x = x.to(self.device)
            output = self(x)
            output = output.cpu().numpy()[0]  # Get first prediction

            # Extract and validate parameters
            vol = output[0]
            skew = np.clip(output[1], -0.99, 0.99)
            df = np.clip(output[2], 2.05, 300.0)

            # Calculate VaR
            dist = self.dist()
            ppf_value = dist.ppf(0.025, [df, skew])
            var = np.sqrt(vol) * ppf_value

            # Return all parameters in a dictionary
            return {
                'VaR': var,
                'Volatility': vol,
                'Skewness': skew,
                'DegreesOfFreedom': df
            }

In [ ]:
# In your data module or experiment code
def gather_prediction(self, predictions_dict):
    """
    Store all predictions in the dataframe

    Args:
        predictions_dict (dict): Dictionary containing all predictions
    """
    current_position = self.test_case + self.training_length

    # Initialize columns if they don't exist
    for key in predictions_dict.keys():
        if key not in self.df.columns:
            self.df[key] = np.nan

    # Store all predictions
    for key, value in predictions_dict.items():
        self.df.iloc[current_position, self.df.columns.get_loc(key)] = value

    # Print predictions
    print(f"Predictions for step {self.test_case + 1}:")
    for key, value in predictions_dict.items():
        print(f"{key}: {value:.6f}")
    print("-" * 50)

# In your training loop
try:
    predictions = model.predict_var(dm.X_test)

    # Transform VaR back to original scale if needed
    if isinstance(predictions, dict):
        predictions['VaR'] = dm.preprocessing.inverse_transform(
            np.array(predictions['VaR']).reshape(-1, 1)
        )[0][0]
        dm.gather_prediction(predictions)
    else:
        # Handle non-skewed models
        var = dm.preprocessing.inverse_transform(
            np.array(predictions).reshape(-1, 1)
        )[0][0]
        dm.gather_prediction({'VaR': var})

except Exception as e:
    print(f"Error during prediction: {str(e)}")

In [ ]:
import torch
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from torch.optim.lr_scheduler import ReduceLROnPlateau

def experiment_nikkei(model_name='garch_skew'):
    # Set up device and print info
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Model setup with proper configuration
    if model_name == 'garch_skew':
        model_class = SkewedGARCHVaRNet
        dist = SkewStudent
        loss = hansen_garch_skewed_student_loss
    elif model_name == 'garch_norm':
        model_class = GARCHVaRNet
        dist = norm
        loss = garch_normal_loss
    elif model_name == 'caviar':
        model_class = VaRNet
        dist = None
        loss = caviar_loss
    elif model_name == 'caviar_huber':
        model_class = VaRNet
        dist = None
        loss = huber_loss
    else:
        raise ValueError(f"Unknown model name: {model_name}")

    # Download and prepare Nikkei data
    nikkei = yf.download('^N225', start='2000-01-01')
    data = nikkei[['Close']].copy()
    data['log_returns'] = np.log(data['Close'] / data['Close'].shift(1))
    data = data.dropna()
    data['VaR'] = np.nan

    # Training parameters setup
    memory_sizes = [20]
    total_samples = len(data)
    sample_points = [int(total_samples * 0.7)]
    sample_dates = [data.index[point].strftime('%Y-%m-%d') for point in sample_points]

    class DeviceAwareDataModule(ValueAtRiskDataModule):
        def __init__(self, *args, device=None, **kwargs):
            super().__init__(*args, **kwargs)
            self.device = device if device is not None else torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        def setup_train(self):
            super().setup_train()
            if self.X_train is not None:
                self.X_train = self.X_train.to(self.device)
            if self.y_train is not None:
                self.y_train = self.y_train.to(self.device)
            if self.X_test is not None:
                self.X_test = self.X_test.to(self.device)

    # Add LR scheduler to model
    class ModelWithScheduler(model_class):
        def configure_optimizers(self):
            optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
            scheduler = {
                'scheduler': ReduceLROnPlateau(optimizer,
                                             mode='min',
                                             factor=0.1,
                                             patience=5,
                                             verbose=True),
                'monitor': 'train_loss',  # Monitors training loss
                'interval': 'epoch',
                'frequency': 1
            }
            return [optimizer], [scheduler]

    # Training loop
    for mem_size in memory_sizes:
        # Training parameters
        p = dict(
            training_length=1000,
            seq_len=mem_size,
            batch_size=128,
            criterion=loss,
            max_epochs=100,
            n_features=1,
            hidden_size=100,
            num_layers=1,
            dropout=0,
            learning_rate=3e-4,
            num_train=100
        )

        for sample_start in sample_dates:
            # Prepare data for current sample
            current_data = data.loc[(data.index > sample_start)].copy()

            # Initialize data module
            dm = DeviceAwareDataModule(
                df=current_data[['log_returns', 'VaR']],
                training_length=p['training_length'],
                seq_len=p['seq_len'],
                batch_size=p['batch_size'],
                num_workers=8,
                device=device
            )

            for test_case in range(p['num_train']):
                print(f"Processing test case {test_case + 1}/{p['num_train']}")

                # Initialize and configure model with scheduler
                model = ModelWithScheduler(
                    n_features=p['n_features'],
                    hidden_size=p['hidden_size'],
                    seq_len=p['seq_len'],
                    batch_size=p['batch_size'],
                    criterion=p['criterion'],
                    num_layers=p['num_layers'],
                    dropout=p['dropout'],
                    learning_rate=p['learning_rate'],
                    dist=dist
                ).to(device)

                # Configure callbacks
                early_stopping = EarlyStopping(
                    monitor='train_loss',
                    patience=13,
                    verbose=True,
                    mode='min'
                )

                lr_monitor = LearningRateMonitor(logging_interval='epoch')

                # Configure trainer with callbacks
                trainer = Trainer(
                    max_epochs=p['max_epochs'],
                    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
                    devices=1,
                    enable_progress_bar=True,
                    log_every_n_steps=1,
                    enable_model_summary=True,
                    enable_checkpointing=False,
                    callbacks=[early_stopping, lr_monitor]  # Add callbacks
                )

                # Setup and train
                dm.setup_train()
                trainer.fit(model, dm)

                # Make predictions
                try:
                    with torch.no_grad():
                        prediction = model.predict_var(dm.X_test)

                        # Ensure prediction is properly formatted
                        if isinstance(prediction, torch.Tensor):
                            prediction = prediction.cpu().numpy()
                        prediction = np.array(prediction).reshape(-1, 1)

                        # Transform prediction back to original scale
                        transformed_prediction = dm.preprocessing.inverse_transform(prediction)[0][0]

                        # Store prediction
                        dm.gather_prediction(transformed_prediction)

                except Exception as e:
                    print(f"Error during prediction: {str(e)}")
                    continue

                # Move to next timestep
                dm.move_timestep()

            # Save results for current sample
            output_filename = f'nikkei_{model_name}_{sample_start}_{mem_size}.csv'
            dm.df[['log_returns', 'VaR']].to_csv(output_filename)
            print(f"Results saved to {output_filename}")

if __name__ == "__main__":

    experiment_nikkei('garch_skew')